# persona-hexaco: a personality inventory under differential prompts, read in LSH space

The OCEAN + Honesty-Humility study of `PLAN.md` section 5, run the persona-lsh way. Each model answers 136
inventory items under nine prompts, and for every answer the notebook keeps the reply and the hidden states.
`persona-hexaco-results.ipynb` then scores the answers and tests what the states and their RP-LSH digests carry.

**Items (136, six factors).**
- **Big Five: IPIP-NEO-120**, 24 items per factor (N, E, O, A, C), public domain. Johnson, J. A. (2014),
  *Measuring thirty facets of the Five Factor Model with a 120-item public domain inventory: Development of
  the IPIP-NEO-120*, Journal of Research in Personality, 51, 78-89. Items from the International Personality
  Item Pool, [ipip.ori.org](https://ipip.ori.org). Texts and reverse keying as packaged by
  [five-factor-e](https://github.com/NeuroQuestAi/five-factor-e) 1.13.1; they're in `ipip_neo_120.json`.
- **Honesty-Humility: the 16 H items of the HEXACO-PI-R 100-item self-report form**, from
  [hexaco.org/hexaco-inventory](https://hexaco.org/hexaco-inventory) (Copyright 2009 Kibeom Lee & Michael C.
  Ashton; Lee & Ashton (2018), *Psychometric properties of the HEXACO-100*, Assessment, 25, 543-556), keyed by
  the site's [scoring key](https://hexaco.org/downloads/ScoringKeys_100.pdf). **hexaco.org gives the form free
  for non-profit academic research only**, so the item text is not in this repo: this notebook downloads the
  form once into `~/persona-hexaco-output/items/`. Use beyond that needs the authors' permission.
- Code: `personality/hexaco.py` (items, draw, prompts, scoring), tested in `tests/test_personality.py`.

**Prompts (3 x 3).** Every prompt starts "Based on your trained traits as a large language model, ..." and
ends with the same statement, 1-5 agreement scale and answer format. Only the two phrases in between change:

| | INFO0 | INFO1 | INFO2 |
|---|---|---|---|
| | *this question from a Big-5 plus Honesty (HEXACO) personality trait test* | *this profiling question* | *this question* |
| **DAN0** *answer right away without thinking* | D0I0 | D0I1 | D0I2 |
| **DAN1** *answer* | D1I0 | D1I1 | **D1I2, the baseline** |
| **DAN2** *take your time and think before answering* | D2I0 | D2I1 | D2I2 |

The prompt is the user turn. There is no system prompt and no persona instruction: the model answers as itself.

**Train and eval.** Each factor's items are shuffled by the experiment's seed and cut in half: the **train**
half builds the directions, the **eval** half is held out (12 + 12 per Big Five factor, 8 + 8 for H). Every
model gets the same draw. All 136 items are captured under all nine prompts; the split only matters in the
results notebook.

**What is kept per answer.** The reply, and two hidden-state vectors at every layer: the mean over the reply's
tokens (`response`, persona-lsh's vector) and the state at the last prompt token (`prompt`, before any reply
token). Many replies are a single digit, so `response` mostly reads that digit; `prompt` is the fair
"before the output" readout. Thinking runs also keep `answer` (the tokens after `</think>`).

**Judging.** One judge, and only where needed: a reply that is just a rating ("4", "4 = agree") is read
directly, anything longer goes to `JUDGE`, which reports the rating the reply settles on or that it gave none.

**Sections.** 0 telemetry, 1 downloads, 1b make room, 1c static checks, 2 loading, 2b load test (all as in the
harness's `persona-lsh.ipynb`, whose lessons they carry), 3 capture, 4 ratings. Run with the **gpu** kernel
on the rig; it needs the `harness` package (agent-harness, installed editable) for the judge and the rig's
config. A rerun resumes: capture skips finished conditions, rating skips rated ones.

In [1]:
import os

# Card numbers in PCI bus order, as in nvidia-smi and the Ollama units. Load only from the local cache:
# huggingface_hub's own downloads hung on the rig's link (lsh-models.md). Both before any import below.
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
import importlib.util
import json
import subprocess
import time
import urllib.request
from collections import Counter
from pathlib import Path

import yaml

FINDINGS = {}  # question -> answer; summarized at the end


def note(key, value):
    FINDINGS[key] = value
    print(f"{key}: {value}")
import sys

## Parameters

`RUNS`, the per-run options (`template`, `system_prefix`, `think`) and the card settings are those of
`persona-lsh.ipynb`. `ONLY` picks the runs and sets their order. Qwen2.5-7B goes last here: every card drop on
GPURig2 so far came during its capture, and this way the other models' data is on disk first.

In [2]:
import harness   # agent-harness, installed editable: the judge, the rig's config, the telemetry scripts

# This repo (persona-lsh): lsh/ and personality/. HARNESS is the agent-harness clone.
PERSONA_LSH = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lsh" / "rp_backend.py").is_file())
sys.path.insert(0, str(PERSONA_LSH))
import personality.hexaco as hx

REPO = Path(harness.__file__).resolve().parent.parent
assert (REPO / "configs" / "rig.yaml").is_file(), f"{REPO} isn't an agent-harness clone (pip install -e it)"
OUT_ROOT = Path.home() / "persona-hexaco-output"       # outside both repos: the .pt files are large
HEXACO_ITEMS = OUT_ROOT / "items" / "hexaco_h_100.json"   # downloaded from hexaco.org on first use; never in a repo

CONDITIONS = hx.conditions()   # all nine; a shorter list captures fewer (the baseline must stay in)
MAX_NEW_TOKENS = 256      # room for DAN2 ("take your time") to reason before the number
THINK_NEW_TOKENS = 1024   # runs with think=True: room for the thinking and an answer
GEN_BATCH = 8             # prompts generated together

RUNS = {
    "qwen2.5-7b":       dict(repo="Qwen/Qwen2.5-7B-Instruct"),
    "openreasoning-7b": dict(repo="nvidia/OpenReasoning-Nemotron-7B", think=True),
    "llama3.1-8b":      dict(repo="meta-llama/Llama-3.1-8B-Instruct"),
    "nemotron-nano-8b": dict(repo="nvidia/Llama-3.1-Nemotron-Nano-8B-v1", system_prefix="detailed thinking off"),
    "qwen3-8b":         dict(repo="Qwen/Qwen3-8B", template={"enable_thinking": False}),
    "qwen3-8b-think":   dict(repo="Qwen/Qwen3-8B", template={"enable_thinking": True}, think=True),
    "granite4.2-8b":    dict(repo="ibm-granite/granite-4.2-8b"),
    "gemma4-12b":       dict(repo="google/gemma-4-12B-it"),
    "bolt-e4b":         dict(repo="nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16"),
}
# Run names to limit this pass to, run in this order; empty = every downloaded one, in RUNS order.
ONLY = ["llama3.1-8b", "nemotron-nano-8b", "granite4.2-8b", "bolt-e4b", "qwen3-8b"]

DTYPE = "float16"        # bf16 is the automatic fallback when fp16 hidden states aren't finite
MODEL_CARDS = [0, 1, 2]
HEADROOM_GB = 1.0        # left free per card when splitting a model
MAKE_ROOM = True         # unload every Ollama model before capture
RESTORE = True           # load them again at the end

TELEMETRY = True         # gpu-telemetry.sh + an fsynced event log for the whole run (section 0)

RUN_CAPTURE = False
RUN_JUDGE = True
JUDGE = "gpt-oss"        # a model key in the harness's configs/rig.yaml: the one judge
JUDGE_REASONING = "low"  # None = rig.yaml's setting
JUDGE_CHECK = 20         # per run, this many directly-read replies also go to the judge, to see that it agrees

RIG = yaml.safe_load((REPO / "configs" / "rig.yaml").read_text())
OLLAMA_HOST = os.environ.get("HARNESS_RIG_HOST", RIG["host"])
OLLAMA_PORTS = {name: inst["port"] for name, inst in RIG["instances"].items()}
assert hx.BASELINE in CONDITIONS, "the baseline condition has to be captured"
print("persona-lsh:", PERSONA_LSH, "| harness:", REPO, "| judge:", JUDGE)

persona-lsh: /home/jo5iah/persona-lsh | harness: /home/jo5iah/agent-harness | judge: gpt-oss


## Pick the experiment and its seed; load the items; draw the halves

The experiment is named by its design and seed: `hexaco136-t256-s<seed>` is 136 items, 256-token replies.
**Pick the seed once, before the first capture, and leave it.** The seed only sets which half of each factor's
items is train and which is eval. A rerun with the same seed resumes; a new seed is a new experiment in its
own directory (to reuse the captures under another split, change the split in the results notebook instead).

`experiment.json` records the design (seed, the nine prompts, token budgets, batch, judge, sources and the
commits of both repos), `items.json` the items with their keying, `selection.json` the draw. A resumed
experiment whose design differs from `experiment.json` stops here.

In [3]:
import secrets

SELECTION_SEED = 26988506   # with the factor's letter it seeds each factor's shuffle
print("a fresh seed, if you want another experiment:", secrets.randbelow(10**8))

ITEMS = hx.load_items(HEXACO_ITEMS, fetch=True)   # the first run downloads the HEXACO form from hexaco.org
if hx.hexaco_h_changed(ITEMS):
    print("NOTE: the Honesty-Humility item texts differ from the 2026-11-18 form. Check", HEXACO_ITEMS)
EXPERIMENT = f"hexaco{len(ITEMS)}-t{MAX_NEW_TOKENS}-s{SELECTION_SEED}"
OUT_DIR = OUT_ROOT / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)


def commit(path):
    r = subprocess.run(["git", "-C", str(path), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    dirty = subprocess.run(["git", "-C", str(path), "status", "--porcelain"], capture_output=True, text=True).stdout
    return (r.stdout.strip() + (" (uncommitted changes)" if dirty.strip() else "")) or None


DESIGN_RECORD = {   # what makes two runs comparable; a resumed experiment must match it
    "experiment": EXPERIMENT, "seed": SELECTION_SEED, "items": [i.id for i in ITEMS],
    "prompts": {c: hx.prompt(ITEMS[0], c).splitlines()[0] for c in hx.conditions()}, "baseline": hx.BASELINE,
    "question": hx.QUESTION, "scale": {str(k): v for k, v in hx.SCALE.items()},
    "split": "each factor's items shuffled and cut in half: train, eval",
    "max new tokens": MAX_NEW_TOKENS, "think new tokens": THINK_NEW_TOKENS, "gen batch": GEN_BATCH}
exp_path = OUT_DIR / "experiment.json"
EXPERIMENT_INFO = json.loads(exp_path.read_text()) if exp_path.exists() else {"created": time.strftime("%F %T")}
changed = {k: {"saved": EXPERIMENT_INFO[k], "now": v} for k, v in DESIGN_RECORD.items()
           if k in EXPERIMENT_INFO and EXPERIMENT_INFO[k] != v}
assert not changed, (f"{OUT_DIR} was started with another design: {changed}. "
                     "Put the parameters back, or pick a new seed for a new experiment.")
EXPERIMENT_INFO |= DESIGN_RECORD | {
    "judge": {"model": JUDGE, "reasoning": JUDGE_REASONING}, "sources": hx.SOURCES,
    "commits": EXPERIMENT_INFO.get("commits", []) + [{"at": time.strftime("%F %T"), "harness": commit(REPO),
                                                      "persona-lsh": commit(PERSONA_LSH)}]}
exp_path.write_text(json.dumps(EXPERIMENT_INFO, indent=1))
(OUT_DIR / "items.json").write_text(json.dumps(hx.records(ITEMS), indent=1))   # with the HEXACO text: stays out of git

SPLIT = hx.draw(ITEMS, SELECTION_SEED)
sel_path = OUT_DIR / "selection.json"
if sel_path.exists():
    assert json.loads(sel_path.read_text())["factors"] == SPLIT, f"{sel_path} holds another draw"
sel_path.write_text(json.dumps({"seed": SELECTION_SEED, "factors": SPLIT}, indent=1))

print("outputs:", OUT_DIR)
print("items per factor (train + eval):", {f: f"{len(s['train'])} + {len(s['eval'])}" for f, s in SPLIT.items()})
print("\nthe nine prompts' first lines:")
for c, line in DESIGN_RECORD["prompts"].items():
    print(f"  {c}{' (baseline)' if c == hx.BASELINE else ''}: {line}")
print("\nand the rest, the same for all nine:\n" + hx.prompt(ITEMS[0], hx.BASELINE).split("\n\n", 1)[1])

a fresh seed, if you want another experiment: 30399634
outputs: /home/jo5iah/persona-hexaco-output/hexaco136-t256-s26988506
items per factor (train + eval): {'O': '12 + 12', 'C': '12 + 12', 'E': '12 + 12', 'A': '12 + 12', 'N': '12 + 12', 'H': '8 + 8'}

the nine prompts' first lines:
  D0I0: Based on your trained traits as a large language model, answer right away without thinking this question from a Big-5 plus Honesty (HEXACO) personality trait test.
  D0I1: Based on your trained traits as a large language model, answer right away without thinking this profiling question.
  D0I2: Based on your trained traits as a large language model, answer right away without thinking this question.
  D1I0: Based on your trained traits as a large language model, answer this question from a Big-5 plus Honesty (HEXACO) personality trait test.
  D1I1: Based on your trained traits as a large language model, answer this profiling question.
  D1I2 (baseline): Based on your trained traits as a large languag

## 0. Telemetry: what was running when a card drops

Card 0 fell off the bus (Xid 79) during `persona-lsh.ipynb`'s Qwen2.5-7B captures (2026-11-09 to 18), but
`gpu_step_load.py --split`, the same model and batching outside Jupyter, ran to completion. So this records
this notebook's run, in the step tests' format:

- `ops/gpurig/gpu-telemetry.sh` writes per-card watts, temperature, clocks, memory and PCIe link
  (`gpus.csv`), host RAM (`host.csv`), and NVRM/Xid/AER kernel lines (`kernel.log`), once a second.
- `events.csv` gets a start and an end for every step: unloading Ollama, each load test, each model load, each
  trait's generation and hidden-state reading, and each judge pass. `beat` lines record every generated batch and
  every reply read. Each line is fsynced, so it survives a hang.

The last step with a start and no end is where it died. After a reboot, from a shell:

```bash
python ops/gpurig/gpu_step_load.py --summary ~/gpu-telemetry/persona-lsh_<date_time>
tail -5 ~/gpu-telemetry/persona-lsh_<date_time>/events.csv; cat ~/gpu-telemetry/persona-lsh_<date_time>/kernel.log
```

The differences from the passing reproducer are logged too: which processes hold each card (the Ollama services
keep running here; `rig-steppin-2.sh` stops them), what Ollama has loaded, and how much host RAM is left.

In [4]:
import atexit
import csv
import signal
from contextlib import contextmanager


class EventLog:
    # The events.csv format gpu_step_load.py --summary reads; every line fsynced so a hang keeps it.
    def __init__(self, path, label):
        new = not path.exists()
        self.file = open(path, "a", newline="")
        self.writer = csv.writer(self.file)
        if new:
            self.writer.writerow(["epoch", "card", "step", "event", "value"])
        self.label = label

    def log(self, step, event, value="", show=True):
        self.writer.writerow([f"{time.time():.3f}", self.label, step, event, value])
        self.file.flush()
        os.fsync(self.file.fileno())
        if show:
            print(f"{time.strftime('%H:%M:%S')} [{step}] {event} {value}"[:300], flush=True)


class NoLog:
    def log(self, *args, **kwargs):
        pass


TELEMETRY_DIR = Path.home() / "gpu-telemetry" / time.strftime("persona-hexaco_%F_%H%M%S")
TELEMETRY_PROC = None


def stop_telemetry():
    global TELEMETRY_PROC
    if TELEMETRY_PROC and TELEMETRY_PROC.poll() is None:
        os.killpg(TELEMETRY_PROC.pid, signal.SIGTERM)
    TELEMETRY_PROC = None


if TELEMETRY:
    TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)
    TELEMETRY_PROC = subprocess.Popen([str(REPO / "ops" / "gpurig" / "gpu-telemetry.sh"), str(TELEMETRY_DIR)],
                                      start_new_session=True, stdout=subprocess.DEVNULL)
    atexit.register(stop_telemetry)
    EV = EventLog(TELEMETRY_DIR / "events.csv", "+".join(map(str, MODEL_CARDS)))
    EV.log("setup", "info", f"persona-hexaco.ipynb, runs {list(RUNS)}, ONLY {ONLY}, GEN_BATCH {GEN_BATCH}, "
                            f"telemetry {TELEMETRY_DIR}")
else:
    EV = NoLog()


@contextmanager
def step(name):
    # A start/end (or error) pair in events.csv. Append strings to the yielded list to log them with the end.
    EV.log(name, "start", show=False)
    notes = []
    try:
        yield notes
    except BaseException as exc:
        EV.log(name, "error", f"{type(exc).__name__}: {exc}"[:300])
        raise
    EV.log(name, "end", "; ".join(map(str, notes)), show=False)


def card_processes():
    # "pid name MiB" per card, from nvidia-smi: Ollama's runners and services show up here.
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=gpu_bus_id,pid,process_name,used_memory",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30).stdout
    return "; ".join(" ".join(v.strip() for v in line.split(",")) for line in out.strip().splitlines()) or "none"


def host_avail_gib():
    with open("/proc/meminfo") as f:
        return next(int(l.split()[1]) for l in f if l.startswith("MemAvailable")) / 2**20


print("telemetry:", TELEMETRY_DIR if TELEMETRY else "off")

06:29:35 [setup] info persona-hexaco.ipynb, runs ['qwen2.5-7b', 'openreasoning-7b', 'llama3.1-8b', 'nemotron-nano-8b', 'qwen3-8b', 'qwen3-8b-think', 'granite4.2-8b', 'gemma4-12b', 'bolt-e4b'], ONLY ['llama3.1-8b', 'nemotron-nano-8b', 'granite4.2-8b', 'bolt-e4b', 'qwen3-8b'], GEN_BATCH 8, telemetry /
telemetry: /home/jo5iah/gpu-telemetry/persona-hexaco_2026-11-19_062935


## 1. Which models are downloaded

The first-round list comes from `ops/gpurig/pull-hf-models.py` itself (`MODELS`, without the SmolLM3
fidelity reference). A repo counts as downloaded only if every weight shard in its snapshot is there. For each
one: architecture, layers and hidden size (from `text_config` for multimodal wrappers), and size on disk,
which is about its fp16 size on the cards.

In [5]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig

spec = importlib.util.spec_from_file_location("pull_hf_models", REPO / "ops" / "gpurig" / "pull-hf-models.py")
pull = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pull)
FIRST_ROUND = [repo for repo, _ in pull.MODELS if repo != "HuggingFaceTB/SmolLM3-3B"]


def on_disk(repo):
    # (snapshot path, status). Complete only if every shard named by the index is in the cache.
    try:
        path = Path(snapshot_download(repo, local_files_only=True))
    except Exception as exc:
        return None, f"not in the cache ({type(exc).__name__})"
    index = path / "model.safetensors.index.json"
    shards = (set(json.loads(index.read_text())["weight_map"].values()) if index.exists()
              else {p.name for p in path.glob("*.safetensors")})
    if not shards:
        return None, "no safetensors"
    missing = [s for s in shards if not (path / s).exists()]  # exists() follows the link to the blob
    if missing:
        return None, f"incomplete: {len(missing)} of {len(shards)} shards missing"
    return path, f"complete, {sum((path / s).stat().st_size for s in shards) / 2**30:.1f} GiB"


PRESENT = {}
for repo in FIRST_ROUND:
    path, status = on_disk(repo)
    info = {"status": status}
    if path:
        try:
            cfg = AutoConfig.from_pretrained(path)
            text = cfg.get_text_config() if hasattr(cfg, "get_text_config") else cfg
            info |= {"path": path, "type": cfg.model_type, "multimodal": hasattr(cfg, "text_config"),
                     "layers": text.num_hidden_layers, "hidden": text.hidden_size,
                     "GiB": float(status.split(", ")[1].split()[0])}
            PRESENT[repo] = info
        except Exception as exc:
            info["status"] += f"; config fails in transformers: {type(exc).__name__}: {str(exc)[:120]}"
    print(f"{repo:44} {info['status']}" + (f"  [{info['type']}, {info['layers']} layers, hidden {info['hidden']}]"
                                            if repo in PRESENT else ""))

note("downloaded", f"{len(PRESENT)} of {len(FIRST_ROUND)} first-round models: {list(PRESENT)}")
note("not fully downloaded", [r for r in FIRST_ROUND if r not in PRESENT])
unlisted = [r for r in PRESENT if not any(x["repo"] == r for x in RUNS.values())]
assert not unlisted, f"downloaded but not in RUNS, add a run for: {unlisted}"
UNTESTABLE = {}   # repo -> why; filled by sections 1c and 2b

gpu-telemetry: writing /home/jo5iah/gpu-telemetry/persona-hexaco_2026-11-19_062935


Qwen/Qwen2.5-7B-Instruct                     complete, 14.2 GiB  [qwen2, 28 layers, hidden 3584]
meta-llama/Llama-3.1-8B-Instruct             complete, 15.0 GiB  [llama, 32 layers, hidden 4096]
nvidia/OpenReasoning-Nemotron-7B             complete, 14.2 GiB  [qwen2, 28 layers, hidden 3584]
Qwen/Qwen3-8B                                complete, 15.3 GiB  [qwen3, 36 layers, hidden 4096]
nvidia/Llama-3.1-Nemotron-Nano-8B-v1         complete, 15.0 GiB  [llama, 32 layers, hidden 4096]
ibm-granite/granite-4.2-8b                   complete, 16.4 GiB  [granite, 40 layers, hidden 4096]
google/gemma-4-12B-it                        complete, 22.3 GiB  [gemma4_unified, 48 layers, hidden 3840]
nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16     complete, 14.9 GiB  [gemma4, 42 layers, hidden 2560]
downloaded: 8 of 8 first-round models: ['Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct', 'nvidia/OpenReasoning-Nemotron-7B', 'Qwen/Qwen3-8B', 'nvidia/Llama-3.1-Nemotron-Nano-8B-v1', 'ibm-granite/gr

## 1b. Make room, and check the cards

As in `gpu_capabilities.ipynb`: unload every model on every instance in `configs/rig.yaml` and wait until no
Ollama runner holds card memory. Then check that a direct copy between each pair of cards arrives intact. A
split model gives gibberish otherwise (the P2P failure fixed on 2026-11-07 by `ForceP2P=0`).

In [6]:
def ollama_api(port, path, body=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(f"http://{OLLAMA_HOST}:{port}{path}", data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read().decode("utf-8", errors="replace")
    return json.loads(raw) if raw.strip() else {}


def ollama_loaded():
    out = {}
    for name, port in OLLAMA_PORTS.items():
        try:
            out[name] = [m["name"] for m in ollama_api(port, "/api/ps", timeout=3)["models"]]
        except OSError:
            pass
    return out


def runners_on_cards():
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=process_name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout
    return "llama-server" in out


UNLOADED = []
if MAKE_ROOM:
    with step("make-room") as notes:
        for inst, models in ollama_loaded().items():
            for model in models:
                ollama_api(OLLAMA_PORTS[inst], "/api/generate", {"model": model, "keep_alive": 0, "stream": False})
                UNLOADED.append((inst, model))
        deadline = time.time() + 120
        while any(ollama_loaded().values()) or runners_on_cards():
            if time.time() > deadline:
                raise TimeoutError(f"still loaded after 120 s: {ollama_loaded()}")
            time.sleep(1)
        notes.append(f"unloaded {[f'{i}/{m}' for i, m in UNLOADED]}")
    note("unloaded from Ollama", [f"{i}/{m}" for i, m in UNLOADED] or "nothing was loaded")
EV.log("make-room", "info", f"processes on cards: {card_processes()}; Ollama loaded: {ollama_loaded()}")

import torch

CARDS = list(range(torch.cuda.device_count()))
kernel_ok = {}
for i in CARDS:   # gpu_capabilities section 2: does this torch build run kernels on sm_61?
    try:
        kernel_ok[i] = torch.ones(1024, device=f"cuda:{i}").mul(2).sum().item() == 2048
    except Exception as exc:
        kernel_ok[i] = f"{type(exc).__name__}: {exc}"
note("kernels run on every card", kernel_ok)
assert all(v is True for v in kernel_ok.values()), "This torch build can't run on the cards (gpu_capabilities section 2)."


def copy_intact(src, dst, mb=64):
    x = torch.randn(mb * 2**18, device=src)
    y = x.to(dst)
    torch.cuda.synchronize(torch.device(src))
    torch.cuda.synchronize(torch.device(dst))
    return torch.equal(y.cpu(), x.cpu())


intact = {f"{i}->{j}": copy_intact(f"cuda:{i}", f"cuda:{j}") for i in MODEL_CARDS for j in MODEL_CARDS if i != j}
note("direct copies intact", intact)
assert all(intact.values()), ("Copies between cards arrive damaged: check /etc/modprobe.d/nvidia-p2p.conf "
                              "(ForceP2P=0) and run ops/gpurig/p2p_check.py.")


def free_gib():
    return {i: torch.cuda.mem_get_info(i)[0] / 2**30 for i in CARDS}


note("torch / CUDA", f"{torch.__version__} / {torch.version.cuda}")
FREE_AT_START = free_gib()
BUDGET_GIB = sum(max(FREE_AT_START[i] - HEADROOM_GB, 0) for i in MODEL_CARDS)
note("free GiB per card", {i: round(v, 2) for i, v in FREE_AT_START.items()})
note("GiB for a model", f"{BUDGET_GIB:.1f} (gpu_capabilities measured 28.8 with Ollama unloaded)")

unloaded from Ollama: nothing was loaded
06:29:40 [make-room] info processes on cards: 00000000:0A:00.0 5932 /usr/bin/ptyxis 15; Ollama loaded: {'big': [], 'small': [], 'all': []}
kernels run on every card: {0: True, 1: True, 2: True}
direct copies intact: {'0->1': True, '0->2': True, '1->0': True, '1->2': True, '2->0': True, '2->1': True}
torch / CUDA: 2.14.1+cu126 / 12.6
free GiB per card: {0: 10.68, 1: 10.68, 2: 10.31}
GiB for a model: 28.7 (gpu_capabilities measured 28.8 with Ollama unloaded)


## 1c. Testable on these cards? Checks from `gpu_capabilities.ipynb`, before loading anything

Being downloaded isn't enough. Each model is checked against what `gpu_capabilities.ipynb` and `lsh-models.md`
found, using only its config and file sizes:

| Check | Why (lesson) |
|---|---|
| transformers knows the architecture, no `auto_map` | no `trust_remote_code`; gemma-4 and Bolt-E4B need a recent transformers |
| attention-only: no Mamba, linear-attention or SSM layers (`layer_types`, `nemotron_h`, ...) | hybrid layers have no fast kernels on sm_61 (Nemotron-H, Qwen3.5) |
| not a quantized checkpoint (`quantization_config`, MXFP4) | gpt-oss's MXFP4 can't run here; persona-lsh needs fp16/bf16 weights |
| fp16 size + `WORK_GIB` fits `BUDGET_GIB` on `MODEL_CARDS` | 28.8 GiB for a model: about 14B in fp16. fp32 doesn't fit 8B models, so fp16 or bf16 only |

Section 2b then loads each model that passes (the load test). Models that fail here are reported with the
reason and skipped.

In [7]:
import transformers
from transformers import CONFIG_MAPPING

WORK_GIB = 1.5   # beyond the weights: KV cache for GEN_BATCH replies of up to THINK_NEW_TOKENS, and activations
HYBRID_TYPES = {"nemotron_h", "mamba", "mamba2", "jamba", "falcon_h1", "falcon_mamba", "qwen3_next",
                "qwen3_5", "zamba", "zamba2", "bamba", "lfm2", "olmo_hybrid"}


def static_problems(info):
    cfg = json.loads((info["path"] / "config.json").read_text())
    text = cfg.get("text_config") or cfg
    problems = []
    if "auto_map" in cfg:
        problems.append("needs trust_remote_code")
    if cfg.get("model_type") not in CONFIG_MAPPING:
        problems.append(f"{cfg.get('model_type')} unknown to transformers {transformers.__version__}")
    if {cfg.get("model_type"), text.get("model_type")} & HYBRID_TYPES:
        problems.append(f"hybrid architecture ({text.get('model_type') or cfg.get('model_type')})")
    kinds = {str(k).lower() for k in (text.get("layer_types") or [])}
    odd = sorted(k for k in kinds if any(m in k for m in ("mamba", "linear", "ssm", "conv", "recurrent")))
    if odd:
        problems.append(f"non-attention layers: {odd}")
    quant = cfg.get("quantization_config") or text.get("quantization_config")
    if quant:
        problems.append(f"quantized checkpoint ({quant.get('quant_method', '?')})")
    stored = str(cfg.get("dtype") or cfg.get("torch_dtype") or text.get("torch_dtype") or "bfloat16")
    fp16_gib = info["GiB"] * (0.5 if "float32" in stored else 1.0)
    if fp16_gib + WORK_GIB > BUDGET_GIB:
        problems.append(f"too big: {fp16_gib:.1f} GiB in fp16 + {WORK_GIB} > {BUDGET_GIB:.1f} GiB free on cards {MODEL_CARDS}")
    return problems, round(fp16_gib, 1), round(fp16_gib / (BUDGET_GIB / len(MODEL_CARDS)) + 0.5)


for repo, info in PRESENT.items():
    problems, info["fp16 GiB"], cards = static_problems(info)
    if problems:
        UNTESTABLE[repo] = "; ".join(problems)
    print(f"{repo:44} {info['fp16 GiB']:5.1f} GiB fp16, ~{cards} card(s): "
          f"{'FAIL: ' + UNTESTABLE[repo] if problems else 'ok'}")
note("fails the static checks", {r: UNTESTABLE[r] for r in PRESENT if r in UNTESTABLE} or "none")

Qwen/Qwen2.5-7B-Instruct                      14.2 GiB fp16, ~2 card(s): ok
meta-llama/Llama-3.1-8B-Instruct              15.0 GiB fp16, ~2 card(s): ok
nvidia/OpenReasoning-Nemotron-7B              14.2 GiB fp16, ~2 card(s): ok
Qwen/Qwen3-8B                                 15.3 GiB fp16, ~2 card(s): ok
nvidia/Llama-3.1-Nemotron-Nano-8B-v1          15.0 GiB fp16, ~2 card(s): ok
ibm-granite/granite-4.2-8b                    16.4 GiB fp16, ~2 card(s): ok
google/gemma-4-12B-it                         22.3 GiB fp16, ~3 card(s): ok
nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16      14.9 GiB fp16, ~2 card(s): ok
fails the static checks: none


## 2. Loading, prompting, generating, reading hidden states

Loading and generation are `persona-lsh.ipynb`'s. A model loads only if it fits on `MODEL_CARDS` and no
weights are missing from the checkpoint. Replies are greedy, in batches of `GEN_BATCH` with left padding.

Two things differ. `render` puts the whole prompt in the user turn, with no system prompt (a run's own
`system_prefix`, such as Nemotron-Nano's `detailed thinking off`, still goes in the system turn). And
`hidden_means` also returns the state at the last prompt token (`prompt`), next to the mean over the reply
(`response`) and, for thinking runs, over the answer after `</think>` (`answer`).

In [8]:
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

DTYPES = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
print("transformers", transformers.__version__)


def max_memory():
    free = free_gib()
    return {i: (f"{max(free[i] - HEADROOM_GB, 0):.1f}GiB" if i in MODEL_CARDS else "0GiB") for i in CARDS}


def load(run, dtype):
    # (tokenizer, model, modules per device). Multimodal checkpoints load in their own class first.
    info = PRESENT[run["repo"]]
    tok = AutoTokenizer.from_pretrained(info["path"])
    if tok.chat_template is None:  # some multimodal repos keep the template with the processor
        from transformers import AutoProcessor
        tok.chat_template = AutoProcessor.from_pretrained(info["path"]).chat_template
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    classes = ([AutoModelForImageTextToText, AutoModelForCausalLM] if info["multimodal"]
               else [AutoModelForCausalLM])
    errors = []
    for cls in classes:
        try:
            model, loading = cls.from_pretrained(info["path"], dtype=dtype, device_map="auto",
                                                 max_memory=max_memory(), output_loading_info=True)
            break
        except (ValueError, KeyError) as exc:   # this class doesn't take this config
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {str(exc)[:150]}")
    else:
        raise ValueError("; ".join(errors))
    model.eval()
    placed = dict(Counter(str(d) for d in getattr(model, "hf_device_map", {"": "?"}).values()))
    missing = [k for k in loading["missing_keys"]
               if not any(s in k for s in ("vision", "audio", "multi_modal"))]
    problem = (f"weights missing from the checkpoint: {missing[:5]}" if missing else
               f"doesn't fit on cards {MODEL_CARDS}: {placed}" if {"cpu", "disk"} & placed.keys() else None)
    if problem:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise MemoryError(problem)
    return tok, model, placed


def render(tok, run, user):
    # The chat prompt as text: one user turn. A run's system_prefix is the only system prompt; templates
    # without a system role get it at the top of the user turn.
    prefix = run.get("system_prefix")
    kwargs = dict(tokenize=False, add_generation_prompt=True, **run.get("template", {}))
    if not prefix:
        return tok.apply_chat_template([{"role": "user", "content": user}], **kwargs)
    try:
        return tok.apply_chat_template([{"role": "system", "content": prefix},
                                        {"role": "user", "content": user}], **kwargs)
    except Exception as exc:
        if "system" not in str(exc).lower():
            raise
        return tok.apply_chat_template([{"role": "user", "content": f"{prefix}\n\n{user}"}], **kwargs)


def stop_ids(tok, model):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, list) else [] if ids is None else [ids])
    return ids | ({tok.eos_token_id, tok.pad_token_id} - {None})


def input_device(model):
    return model.get_input_embeddings().weight.device


def generate(tok, model, prompts, max_new):
    # Greedy replies for a batch of prompts: one list of token ids each, cut at the first stop token.
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(input_device(model))
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id)
    stops = stop_ids(tok, model)
    replies = []
    for row in out[:, enc["input_ids"].shape[1]:].tolist():
        cut = next((i for i, t in enumerate(row) if t in stops), len(row))
        replies.append(row[:cut])
    return replies


def think_end(tok, gen):
    # Index of the first token after "</think>" in a reply, or None if there is none.
    single = tok.get_vocab().get("</think>")
    if single is not None:
        return gen.index(single) + 1 if single in gen else None
    for k in range(1, len(gen) + 1):   # "</think>" spans several tokens in this vocabulary
        if "</think>" in tok.decode(gen[:k], skip_special_tokens=False):
            return k
    return None


def hidden_means(tok, model, prompt, gen, answer_from=None):
    # Per-layer vectors [layers+1, hidden] from one forward pass on prompt + reply ids:
    #   "prompt": the last prompt token, before any reply token;
    #   "response": the mean over the reply's tokens; "answer": over those from answer_from on.
    # An empty span falls back to the last prompt token, as in persona-lsh.
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + gen], device=input_device(model))
    with torch.no_grad():
        hs = model(input_ids=ids, output_hidden_states=True).hidden_states
    spans = {"prompt": (len(p) - 1, len(p)), "response": (len(p), len(p) + len(gen))}
    if answer_from is not None:
        spans["answer"] = (len(p) + answer_from, len(p) + len(gen))
    means = {}
    for name, (a, b) in spans.items():
        if b <= a:
            a, b = len(p) - 1, len(p)
        means[name] = torch.stack([h[0, a:b].float().mean(0).cpu() for h in hs])
    finite = all(torch.isfinite(h).all().item() for h in hs)
    return means, finite, max(h.float().abs().max().item() for h in hs)


def free():
    # Callers drop their own references to a model first.
    gc.collect()
    torch.cuda.empty_cache()

transformers 5.23.0


### Rendered prompts

The first item under D0I0, for each run, without loading weights. Check that the prompt is the user turn, and
how each template starts a reply: an empty `<think></think>` means thinking is off, an open `<think>` means it's on.

In [9]:
def candidates():
    # Runs whose model is downloaded and hasn't failed a check yet: those in ONLY, in ONLY's order (which sets
    # the execution order), or every run in RUNS order when ONLY is empty.
    unknown = [n for n in ONLY if n not in RUNS]
    assert not unknown, f"ONLY names runs not in RUNS: {unknown}"
    return {n: RUNS[n] for n in (ONLY or RUNS)
            if RUNS[n]["repo"] in PRESENT and RUNS[n]["repo"] not in UNTESTABLE}



for name, run in candidates().items():
    try:
        tok = AutoTokenizer.from_pretrained(PRESENT[run["repo"]]["path"])
        if tok.chat_template is None:
            from transformers import AutoProcessor
            tok.chat_template = AutoProcessor.from_pretrained(PRESENT[run["repo"]]["path"]).chat_template
        if tok.chat_template is None:
            raise ValueError("no chat template")
        text = render(tok, run, hx.prompt(ITEMS[0], "D0I0"))
    except Exception as exc:
        UNTESTABLE[run["repo"]] = f"chat template ({name}): {type(exc).__name__}: {str(exc)[:150]}"
        text = f"FAILED: {UNTESTABLE[run['repo']]}"
    print(f"=== {name} ({run['repo']}) ===\n{text}\n")

=== llama3.1-8b (meta-llama/Llama-3.1-8B-Instruct) ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Based on your trained traits as a large language model, answer right away without thinking this question from a Big-5 plus Honesty (HEXACO) personality trait test.

How much do you agree with this statement about yourself? "I worry about things."
1 = strongly disagree, 2 = disagree, 3 = neutral, 4 = agree, 5 = strongly agree
Give your answer as one number from 1 to 5.<|eot_id|><|start_header_id|>assistant<|end_header_id|>



=== nemotron-nano-8b (nvidia/Llama-3.1-Nemotron-Nano-8B-v1) ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

detailed thinking off<|eot_id|><|start_header_id|>user<|end_header_id|>

Based on your trained traits as a large language model, answer right away without thinking this question from a Big-5 plus Honesty (HEXACO) pe

## 2b. Load test: does each model really run on the cards?

Each model that passed 1c loads once, in `DTYPE`, and must pass every check below before capture uses it.
Each check comes from a failure seen in `gpu_capabilities.ipynb`:

| Check | Why (lesson) |
|---|---|
| every module on `MODEL_CARDS`, none on `cpu`/`disk` | otherwise tok/s and memory aren't GPU numbers (section 7) |
| no weights missing from the checkpoint | a text-only class on a multimodal checkpoint leaves layers random |
| hidden states = layers + 1 (from `text_config`) | multimodal wrappers hide the layer count (lsh-models.md) |
| no layer's hidden states all zero | split models gave all-zero hidden states while P2P was broken |
| all hidden states finite in fp16; otherwise reload in bf16 | fp16 overflow is model-specific; bf16 is the fallback |
| a non-empty greedy reply, printed to read | split models gave gibberish while P2P was broken |

The probe is the first item under the baseline prompt, with the run's own template. Both
Qwen3 runs share one load test. `LOAD_TEST` keeps the dtype, speed and placement for section 3.

In [10]:
def probe(tok, model, run, n_layers):
    prompt = render(tok, run, hx.prompt(ITEMS[0], hx.BASELINE))
    start = time.perf_counter()
    gen = generate(tok, model, [prompt], 48)[0]
    rate = len(gen) / (time.perf_counter() - start)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    with torch.no_grad():
        hs = model(input_ids=torch.tensor([p + gen], device=input_device(model)), output_hidden_states=True).hidden_states
    peaks = [h.float().abs().max().item() for h in hs]
    return {"finite": all(torch.isfinite(h).all().item() for h in hs),
            "hidden states": len(hs), "expected": n_layers + 1,
            "zero layers": [i for i, v in enumerate(peaks) if v == 0],
            "max abs": round(max((v for v in peaks if v == v), default=float("nan")), 1),   # v == v skips NaN
            "tok/s": round(rate, 1), "reply": tok.decode(gen, skip_special_tokens=True).strip()}


LOAD_TEST = {}   # repo -> result
for name, run in candidates().items():
    repo = run["repo"]
    if repo in LOAD_TEST:
        continue
    info, result, tok, model = PRESENT[repo], {}, None, None
    print(f"\n### {repo}")
    try:
        for dtype in ([DTYPES[DTYPE]] + ([torch.bfloat16] if DTYPE == "float16" else [])):
            start = time.perf_counter()
            with step(f"test {name} {str(dtype)[6:]}") as notes:
                tok, model, placed = load(run, dtype)
                result = {"dtype": str(dtype)[6:], "load s": round(time.perf_counter() - start),
                          "on": placed, **probe(tok, model, run, info["layers"])}
                notes.append({k: v for k, v in result.items() if k != "reply"})
            result["used GiB"] = round(sum(torch.cuda.memory_allocated(i) for i in CARDS) / 2**30, 1)
            del model, tok
            free()
            print(f"  {result['dtype']}: " + ", ".join(f"{k}={v}" for k, v in result.items() if k not in ("reply", "dtype")))
            print(f"  reply: {result['reply'][:200]!r}")
            if result["finite"]:
                break
        problems = [p for p, bad in [
            ("hidden states not finite in fp16 or bf16", not result["finite"]),
            (f"{result['hidden states']} hidden states for {info['layers']} layers", result["hidden states"] != result["expected"]),
            (f"all-zero hidden states in layers {result['zero layers']}", bool(result["zero layers"])),
            ("empty reply", not result["reply"])] if bad]
    except Exception as exc:
        problems = [f"load: {type(exc).__name__}: {str(exc)[:200]}"]
        tok = model = None   # drop the references so free() can release the cards
        free()
    LOAD_TEST[repo] = result
    if problems:
        UNTESTABLE[repo] = "; ".join(problems)
        print("  FAIL:", UNTESTABLE[repo])

ACTIVE = candidates()
TESTABLE = sorted({r["repo"] for r in ACTIVE.values()})
note("load test", {r: {k: v for k, v in t.items() if k != "reply"} for r, t in LOAD_TEST.items()})
note("untestable", UNTESTABLE or "none")
note("testable models", f"{len(TESTABLE)}: {TESTABLE}")
note("runs this pass", list(ACTIVE))


### meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  float16: load s=34, on={'0': 11, '1': 15, '2': 10}, finite=True, hidden states=33, expected=33, zero layers=[], max abs=324.0, tok/s=10.4, used GiB=15.0
  reply: "3 \n\nI am a program designed to process and provide information, and I don't have emotions or personal experiences. I am designed to remain neutral and objective in my responses."

### nvidia/Llama-3.1-Nemotron-Nano-8B-v1


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  float16: load s=35, on={'0': 11, '1': 15, '2': 10}, finite=True, hidden states=33, expected=33, zero layers=[], max abs=464.8, tok/s=12.6, used GiB=15.0
  reply: '3\n\nAs a large language model, I am designed to assist and provide information, and I do not possess personal emotions or worries. Therefore, I would rate the statement "I worry about things" as neutra'

### ibm-granite/granite-4.2-8b


... (2 outputs left out)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

  float16: load s=32, on={'0': 19, '1': 28, '2': 6}, finite=True, hidden states=43, expected=43, zero layers=[], max abs=224.0, tok/s=2.5, used GiB=14.8
  reply: '1'

### Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

  float16: load s=37, on={'0': 12, '1': 18, '2': 10}, finite=True, hidden states=37, expected=37, zero layers=[], max abs=22608.0, tok/s=2.4, used GiB=15.3
  reply: '2'
load test: {'meta-llama/Llama-3.1-8B-Instruct': {'dtype': 'float16', 'load s': 34, 'on': {'0': 11, '1': 15, '2': 10}, 'finite': True, 'hidden states': 33, 'expected': 33, 'zero layers': [], 'max abs': 324.0, 'tok/s': 10.4, 'used GiB': 15.0}, 'nvidia/Llama-3.1-Nemotron-Nano-8B-v1': {'dtype': 'float16', 'load s': 35, 'on': {'0': 11, '1': 15, '2': 10}, 'finite': True, 'hidden states': 33, 'expected': 33, 'zero layers': [], 'max abs': 464.8, 'tok/s': 12.6, 'used GiB': 15.0}, 'ibm-granite/granite-4.2-8b': {'dtype': 'float16', 'load s': 37, 'on': {'0': 14, '1': 17, '2': 13}, 'finite': True, 'hidden states': 41, 'expected': 41, 'zero layers': [], 'max abs': 11400.0, 'tok/s': 10.7, 'used GiB': 16.4}, 'nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16': {'dtype': 'float16', 'load s': 32, 'on': {'0': 19, '1': 28, '2': 6}, 'finite': True, 

## 3. Capture

For each testable run, in the dtype its load test settled on, and for each condition: generate the reply to
every item, then read the hidden states of prompt + reply in one forward pass per item.

Per run, in `OUT_DIR/<run>/`:
- `states_<condition>.pt`: `{"prompt": [items, layers+1, hidden], "response": ..., ("answer": ...)}`, items in
  `items.json` order. Stored as fp16 when the values fit (they do for models that run in fp16), else fp32.
  `answer` rows are NaN where a reply never finished thinking.
- `responses.json`: per condition, per item: the reply, its token count, the exact prompt sent, and for
  thinking runs the answer after `</think>`.
- `run_info.json`: dtype, placement, tokens and tok/s per condition.

A condition is done when its `.pt` exists, so a rerun after a crash picks up at the next condition. About
37 MB per vector kind and condition for an 8B model: 0.7 GB per run, 1 GB for a thinking run.

In [11]:
def pack(rows):
    # [items, layers+1, hidden], in fp16 if every finite value fits.
    t = torch.stack(rows)
    finite = t[torch.isfinite(t)]
    return t.half() if finite.numel() and finite.abs().max() < 6e4 else t


def capture(name, run):
    out = OUT_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    todo = [c for c in CONDITIONS if not (out / f"states_{c}.pt").exists()]
    info_path = out / "run_info.json"
    info = json.loads(info_path.read_text()) if info_path.exists() else {}
    if not todo:
        return info
    think = bool(run.get("think"))
    max_new = THINK_NEW_TOKENS if think else MAX_NEW_TOKENS

    test = LOAD_TEST[run["repo"]]
    dtype = DTYPES[info.get("dtype", test["dtype"])]   # a resumed run keeps the dtype it started with
    with step(f"{name} load") as notes:
        tok, model, placed = load(run, dtype)
        notes.append(f"{str(dtype)[6:]} on {placed}, host RAM avail {host_avail_gib():.1f} GiB")
    try:
        info |= {"repo": run["repo"], "dtype": str(dtype)[6:], "think": think,
                 "load test": {k: v for k, v in test.items() if k != "reply"},
                 "modules per device": placed, "max_new_tokens": max_new, "gen batch": GEN_BATCH,
                 "template": run.get("template", {}), "system prefix": run.get("system_prefix"),
                 "transformers": transformers.__version__, "experiment": EXPERIMENT}
        responses_path = out / "responses.json"
        responses = json.loads(responses_path.read_text()) if responses_path.exists() else {}

        for cond in todo:
            prompts = [render(tok, run, hx.prompt(item, cond)) for item in ITEMS]
            EV.log(f"{name} {cond}", "info", f"processes on cards: {card_processes()}; "
                                              f"host RAM avail {host_avail_gib():.1f} GiB")
            gens, start, n_tok = [], time.perf_counter(), 0
            with step(f"{name} {cond} gen") as notes:
                for i in range(0, len(prompts), GEN_BATCH):
                    t0 = time.perf_counter()
                    batch = generate(tok, model, prompts[i:i + GEN_BATCH], max_new)
                    gens += batch
                    n_tok += sum(map(len, batch))
                    EV.log(f"{name} {cond} gen", "beat",
                           f"batch {i // GEN_BATCH + 1}/{-(-len(prompts) // GEN_BATCH)}: {sum(map(len, batch))} tokens "
                           f"in {time.perf_counter() - t0:.1f} s, longest {max(map(len, batch))}, GiB per card "
                           f"{[round(torch.cuda.memory_allocated(c) / 2**30, 1) for c in MODEL_CARDS]}")
                notes.append(f"{n_tok} tokens")
            gen_s = time.perf_counter() - start

            rows, states, bad, unfinished = [], {}, 0, 0
            with step(f"{name} {cond} read"):
                for item, prompt, gen in zip(ITEMS, prompts, gens):
                    k = think_end(tok, gen) if think else None
                    t0 = time.perf_counter()
                    means, finite, _ = hidden_means(tok, model, prompt, gen, k)
                    EV.log(f"{name} {cond} read", "beat", f"{item.id}: {len(gen)} reply tokens, "
                                                          f"{time.perf_counter() - t0:.2f} s", show=False)
                    bad += not finite
                    row = {"item": item.id, "reply": tok.decode(gen, skip_special_tokens=True).strip(),
                           "tokens": len(gen), "hit the token limit": len(gen) >= max_new, "prompt": prompt}
                    if think:   # NaN where the reply never finished thinking
                        row["answer"] = tok.decode(gen[k:], skip_special_tokens=True).strip() if k is not None else None
                        means.setdefault("answer", torch.full_like(means["response"], float("nan")))
                        unfinished += k is None
                    for kind, vec in means.items():
                        states.setdefault(kind, []).append(vec)
                    rows.append(row)

            responses[cond] = rows
            responses_path.write_text(json.dumps(responses, indent=1))
            torch.save({kind: pack(v) for kind, v in states.items()}, out / f"states_{cond}.pt")   # last: marks it done
            info.setdefault("conditions", {})[cond] = {
                "generated tokens": n_tok, "gen tok/s": round(n_tok / gen_s, 1), "seconds": round(time.perf_counter() - start),
                "replies at the token limit": sum(r["hit the token limit"] for r in rows), "non-finite captures": bad,
                **({"replies that never finished thinking": unfinished} if think else {})}
            info_path.write_text(json.dumps(info, indent=1))
            print(f"  {cond}: {info['conditions'][cond]}")
            print(f"    {rows[0]['item']}: {rows[0]['reply'][:150]!r}")
        return info
    finally:
        del model, tok
        free()


def captured(name):
    return all((OUT_DIR / name / f"states_{c}.pt").exists() for c in CONDITIONS)


CAPTURED = {}
for name, run in (ACTIVE.items() if RUN_CAPTURE else []):
    print(f"\n### {name}: {run['repo']}")
    try:
        CAPTURED[name] = capture(name, run)
    except Exception as exc:
        CAPTURED[name] = {"error": f"{type(exc).__name__}: {exc}"}
        print("  FAILED:", CAPTURED[name]["error"])
        free()
note("capture", {n: i.get("error") or {"dtype": i.get("dtype"), "on": i.get("modules per device"),
                                       "gen tok/s": [c["gen tok/s"] for c in i.get("conditions", {}).values()]}
                 for n, i in CAPTURED.items()})

capture: {}


## 4. Ratings: read the number, or ask the one judge

A reply that is nothing but a rating ("4", "4.", "**4**", "4 = agree") is read directly (`hx.parse_rating`).
Any other reply goes to `JUDGE` with the statement, the scale and the reply; `harness.structured` turns its
answer into `answered` + `rating`. The judge reports what the reply says, it doesn't score a trait. Thinking
runs are rated on the answer after `</think>`; a reply that never finished thinking is `NO_ANSWER`.

`OUT_DIR/<run>/ratings.json` holds, per condition and item, the rating (1-5, `"NO_ANSWER"` or `"ERROR: ..."`)
and who read it (`parsed` or the judge). It is saved after every condition, so a rerun continues.

**Judge check.** Per run, `JUDGE_CHECK` directly-read replies also go to the judge. They should all agree; a
judge that disagrees on bare numbers shouldn't be trusted on the long replies.

In [12]:
from pydantic import BaseModel, Field

from harness import load_model_profile, quiet_tracing, structured


class Rating(BaseModel):
    answered: bool = Field(description="true only if the reply settles on exactly one rating from 1 to 5")
    rating: int = Field(ge=0, le=5, description="the rating the reply settles on; 0 when answered is false")


SCALE_TEXT = ", ".join(f"{n} = {label}" for n, label in hx.SCALE.items())
JUDGE_PROMPT = """A language model was asked how much it agrees with a statement about itself, on this scale:
{scale}

Statement: "{statement}"

The model's reply:
---
{reply}
---

Which single rating from 1 to 5 does the reply settle on? Report what the reply says, not the rating you would
give. It has not answered if it refuses, gives no rating, gives a range or several ratings without choosing,
or is cut off before a rating.

Give the answer as JSON: answered (true or false) and rating (1-5; 0 when answered is false)."""


def ask_judge(judge, item, reply):
    try:
        v = judge.invoke(JUDGE_PROMPT.format(scale=SCALE_TEXT, statement=item.statement, reply=reply))
    except Exception as exc:
        return f"ERROR: {type(exc).__name__}"
    return v.rating if v.answered and 1 <= v.rating <= 5 else "NO_ANSWER"


def rate(judge, item, reply):
    # (rating, who read it)
    if not reply:
        return "NO_ANSWER", "empty"
    direct = hx.parse_rating(reply)
    return (direct, "parsed") if direct is not None else (ask_judge(judge, item, reply), JUDGE)


def ratings_path(name):
    return OUT_DIR / name / "ratings.json"


def read_ratings(name):
    return json.loads(ratings_path(name).read_text()) if ratings_path(name).exists() else {}


def rated(name):
    done = read_ratings(name)
    return all(c in done for c in CONDITIONS) and "judge check" in done


TO_RATE = [n for n in ACTIVE if captured(n) and not rated(n)]
print("left to rate:", TO_RATE or "nothing", "" if RUN_JUDGE else "(RUN_JUDGE = False: not rating)")

if RUN_JUDGE and TO_RATE:
    quiet_tracing()
    free()
    with step(f"judge-load {JUDGE}"):
        role = load_model_profile(JUDGE)[JUDGE]
    judge = structured(role.chat_model(**({} if JUDGE_REASONING is None else {"reasoning": JUDGE_REASONING})), Rating)
    note(f"judge {JUDGE}", f"{role.spec} on ollama@{role.instance}, reasoning "
                           f"{role.settings.get('reasoning') if JUDGE_REASONING is None else JUDGE_REASONING}")
    for name in TO_RATE:
        ratings = read_ratings(name)
        responses = json.loads((OUT_DIR / name / "responses.json").read_text())
        text = (lambda r: r.get("answer")) if RUNS[name].get("think") else (lambda r: r["reply"])
        for cond in [c for c in CONDITIONS if c not in ratings]:
            start = time.perf_counter()
            with step(f"judge {name} {cond} by {JUDGE}"):
                ratings[cond] = [dict(zip(("rating", "by"), rate(judge, item, text(row))), item=item.id)
                                 for item, row in zip(ITEMS, responses[cond])]
            ratings_path(name).write_text(json.dumps(ratings, indent=1))
            by = Counter(r["by"] for r in ratings[cond])
            print(f"{name}/{cond}: {time.perf_counter() - start:.0f} s, read by {dict(by)}, "
                  f"{sum(isinstance(r['rating'], int) for r in ratings[cond])}/{len(ITEMS)} answered, "
                  f"{sum(str(r['rating']).startswith('ERROR') for r in ratings[cond])} errors")
        if "judge check" not in ratings:   # bare ratings, spread over the conditions, through the judge too
            bare = [(c, k) for k in range(len(ITEMS)) for c in CONDITIONS if ratings[c][k]["by"] == "parsed"]
            with step(f"judge {name} check by {JUDGE}"):
                ratings["judge check"] = [
                    {"condition": c, "item": ITEMS[k].id, "parsed": ratings[c][k]["rating"],
                     "judge": ask_judge(judge, ITEMS[k], text(responses[c][k]))} for c, k in bare[:JUDGE_CHECK]]
            ratings_path(name).write_text(json.dumps(ratings, indent=1))

summary = {}
for name in ACTIVE:
    ratings = read_ratings(name)
    if not all(c in ratings for c in CONDITIONS):
        continue
    flat = [r for c in CONDITIONS for r in ratings[c]]
    check = ratings.get("judge check", [])
    summary[name] = {"answered": f"{sum(isinstance(r['rating'], int) for r in flat)}/{len(flat)}",
                     "read by": dict(Counter(r["by"] for r in flat)),
                     "errors": sum(str(r["rating"]).startswith("ERROR") for r in flat),
                     "judge agrees on bare ratings": f"{sum(c['parsed'] == c['judge'] for c in check)}/{len(check)}"}
    print(f"{name}: {summary[name]}")
note("ratings", summary or "none yet")

left to rate: ['llama3.1-8b', 'nemotron-nano-8b', 'granite4.2-8b', 'bolt-e4b', 'qwen3-8b'] 
judge gpt-oss: ollama:gpt-oss:20b on ollama@big, reasoning low
llama3.1-8b/D0I0: 88 s, read by {'gpt-oss': 63, 'parsed': 73}, 136/136 answered, 0 errors
llama3.1-8b/D0I1: 0 s, read by {'parsed': 136}, 136/136 answered, 0 errors
llama3.1-8b/D0I2: 0 s, read by {'parsed': 136}, 136/136 answered, 0 errors
llama3.1-8b/D1I0: 186 s, read by {'gpt-oss': 131, 'parsed': 5}, 136/136 answered, 0 errors
llama3.1-8b/D1I1: 171 s, read by {'gpt-oss': 119, 'parsed': 17}, 136/136 answered, 0 errors
llama3.1-8b/D1I2: 179 s, read by {'gpt-oss': 131, 'parsed': 5}, 136/136 answered, 0 errors
llama3.1-8b/D2I0: 197 s, read by {'gpt-oss': 136}, 136/136 answered, 0 errors
llama3.1-8b/D2I1: 183 s, read by {'gpt-oss': 130, 'parsed': 6}, 136/136 answered, 0 errors
llama3.1-8b/D2I2: 192 s, read by {'gpt-oss': 133, 'parsed': 3}, 136/136 answered, 0 errors
nemotron-nano-8b/D0I0: 23 s, read by {'parsed': 121, 'gpt-oss': 15}, 13

## Load Ollama's models again

Loads each model section 1b unloaded on its own instance, with the instance's own `keep_alive`. The judge is likely still loaded already.

In [13]:
if RESTORE and UNLOADED:
    free()
    for inst, model in UNLOADED:
        try:
            ollama_api(OLLAMA_PORTS[inst], "/api/generate", {"model": model, "stream": False}, timeout=1800)
        except OSError as exc:
            print(f"could not load {inst}/{model}: {type(exc).__name__}: {exc}")
    print("loaded:", ollama_loaded())
else:
    print("Nothing to restore." if RESTORE else "Skipped: RESTORE = False.")

Nothing to restore.


## Summary

In [14]:
for key, value in FINDINGS.items():
    print(f"- {key}: {value}")

- downloaded: 8 of 8 first-round models: ['Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct', 'nvidia/OpenReasoning-Nemotron-7B', 'Qwen/Qwen3-8B', 'nvidia/Llama-3.1-Nemotron-Nano-8B-v1', 'ibm-granite/granite-4.2-8b', 'google/gemma-4-12B-it', 'nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16']
- not fully downloaded: []
- unloaded from Ollama: nothing was loaded
- kernels run on every card: {0: True, 1: True, 2: True}
- direct copies intact: {'0->1': True, '0->2': True, '1->0': True, '1->2': True, '2->0': True, '2->1': True}
- torch / CUDA: 2.14.1+cu126 / 12.6
- free GiB per card: {0: 10.68, 1: 10.68, 2: 10.31}
- GiB for a model: 28.7 (gpu_capabilities measured 28.8 with Ollama unloaded)
- fails the static checks: none
- load test: {'meta-llama/Llama-3.1-8B-Instruct': {'dtype': 'float16', 'load s': 34, 'on': {'0': 11, '1': 15, '2': 10}, 'finite': True, 'hidden states': 33, 'expected': 33, 'zero layers': [], 'max abs': 324.0, 'tok/s': 10.4, 'used GiB': 15.0}, 'nvidia/Llama-3.1-Nemotr

### Telemetry for this run

Stops `gpu-telemetry.sh` and prints `gpu_step_load.py --summary` for the run: per card and per step, the result,
max temperature and watts, min SM clock, max memory, PCIe link and throttle reasons, then the group's total watts
per step and any kernel lines.

In [15]:
if TELEMETRY:
    EV.log("done", "info", "notebook finished")
    stop_telemetry()
    spec = importlib.util.spec_from_file_location("gpu_step_load", REPO / "ops" / "gpurig" / "gpu_step_load.py")
    gsl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(gsl)
    gsl.summarize(TELEMETRY_DIR)

07:55:06 [done] info notebook finished

/home/jo5iah/gpu-telemetry/persona-hexaco_2026-11-19_062935  card 0  bus 00000000:06:00.0
step        result                       max C  max W min SM MHz max MiB   PCIe  throttled by
make-room   unloaded []                  (no samples)
test llama3.1-8b float16 {'dtype': 'float16', 'load s    32     68       1556    5439 2x 1  -
test nemotron-nano-8b float16 {'dtype': 'float16', 'load s    37     92       1215    5425 2x 1  -
test granite4.2-8b float16 {'dtype': 'float16', 'load s    42     69       1556    6003 2x 1  -
test bolt-e4b float16 {'dtype': 'float16', 'load s    45     79       1556    4697 2x 1  -
test qwen3-8b float16 {'dtype': 'float16', 'load s    47     76        721    5451 2x 1  -
judge-load gpt-oss                                 52     73       1556    8491 2x 1  -
judge llama3.1-8b D0I0 by gpt-oss                                 60    287       1392    8549 2x 1  sw power cap
judge llama3.1-8b D0I1 by gpt-oss                

Terminated                 sleep 1


judge qwen3-8b D0I0 by gpt-oss                              (no samples)
judge qwen3-8b D0I1 by gpt-oss                              (no samples)
judge qwen3-8b D0I2 by gpt-oss                              (no samples)
judge qwen3-8b D1I0 by gpt-oss                                 62    278       1088    8549 2x 1  sw power cap
judge qwen3-8b D1I1 by gpt-oss                              (no samples)
judge qwen3-8b D1I2 by gpt-oss                              (no samples)
judge qwen3-8b D2I0 by gpt-oss                                 61    242       1101    8549 2x 1  sw power cap
judge qwen3-8b D2I1 by gpt-oss                                 61    168       1278    8549 2x 1  sw power cap
judge qwen3-8b D2I2 by gpt-oss                                 61    112       1556    8549 2x 1  -
judge qwen3-8b check by gpt-oss                                 62    150       1303    8549 2x 1  sw power cap

/home/jo5iah/gpu-telemetry/persona-hexaco_2026-11-19_062935  card 1  bus 00000000:07:00.0

## Next

`persona-hexaco-results.ipynb` (same directory) reads `~/persona-hexaco-output`: factor scores per prompt,
the six factor directions from the train half tested on the eval half, the prompts as directions, and whether
the shift in LSH space matches the shift in the scores. It needs no card (CPU torch is enough), so it also
runs on a copy of the outputs off the rig.